# Walkthrough: cash-flow forecaster with spikes and holidays

The pieces behind `scripts/run_pipeline.py`: the simulated corridors, what each rung of the model ladder learns, and the backtest. All core logic lives in `src/`.

In [ ]:
import os, sys
os.environ.setdefault('OMP_NUM_THREADS', '1')
sys.path.insert(0, '../src')

import pandas as pd
from cashflow_forecaster import generate_flows, SeasonalWeekday, SpikeAdjusted, HolidayAdjusted
from cashflow_forecaster.data import TRUE_HOLIDAY_EFFECTS
from cashflow_forecaster.evaluate import backtest_all, ladder_table

## 1. The data
Three corridors, each the product of known parts. Spikes and holiday positions are kept as columns.

In [ ]:
flows = generate_flows()
mx = flows[flows.corridor == 'MX']
mx[mx.date.between('2025-05-05', '2025-05-12')][['date', 'volume', 'holiday_name', 'holiday_offset', 'holiday_factor', 'is_spike']]

## 2. Spikes inflate the baseline
The level of the plain seasonal model vs the spike-cleaned one.

In [ ]:
series = mx[['date', 'volume']]
base = SeasonalWeekday().fit(series)
spike = SpikeAdjusted().fit(series)
print(f'baseline level ${base.level_/1e6:.1f}M, clean level ${spike.level_/1e6:.1f}M')
print(f'spike probability {spike.spike_prob_:.1%}, average spike size x{spike.spike_multiplier_:.2f}')

## 3. Holiday effects learned from history vs the truth

In [ ]:
full = HolidayAdjusted(country='MX').fit(series)
pd.DataFrame({
    'estimated': full.holiday_effects_.round(3),
    'true': {f'{t}:{o}': v for (t, o), v in TRUE_HOLIDAY_EFFECTS.items()},
    'past occurrences': full.holiday_counts_,
})

## 4. The model ladder (walk-forward, 7 days ahead)
Takes about 15 seconds.

In [ ]:
bt = backtest_all(flows, horizons=(7,))
ladder = ladder_table(bt)
money = [c for c in ladder.columns if c not in ('horizon', 'model', 'MSE')]
ladder.assign(**{c: (ladder[c] / 1e6).round(2) for c in money}).drop(columns='MSE')